In [ ]:
# List inside the Volume
files = dbutils.fs.ls("/Volumes/main_udev/ai_labs/aag/artifacts/.internal")

# Pick the newest wheel
latest_dbfs = max(
    files, key=lambda f: f.modificationTime
).path  # e.g., "dbfs:/Volumes/.../thesis-1.0.4-..."
latest_local = latest_dbfs.replace("dbfs:", "")  # -> "/Volumes/.../thesis-1.0.4-..."

print(f"Installing: {latest_local}")
%pip install "$latest_local"
# reinstall even if version is same
%restart_python

In [ ]:
import mlflow

mlflow.set_tracking_uri("databricks")
# Experiments go in Workspace, not Volumes
mlflow.set_experiment(experiment_id="2984249850048933")

In [ ]:
import diffrax
import equinox as eqx  # https://github.com/patrick-kidger/equinox
import jax
import jax.nn as jnn
import jax.numpy as jnp
import jax.random as jr
import matplotlib.pyplot as plt
import numpy as np
import optax  # https://github.com/deepmind/optax
import mlflow
import torch
from pathlib import Path
from torch.utils.data import DataLoader
from thesis.prototyping.dataloader import ParquetDataset
from thesis.prototyping.data_handling import find_parquet_files

mlflow.enable_system_metrics_logging()

In [ ]:
def lipswish(x):
    return 0.909 * jnn.silu(x)

In [ ]:
class VectorField(eqx.Module):
    scale: int | jnp.ndarray
    mlp: eqx.nn.MLP

    def __init__(self, hidden_size, width_size, depth, scale, *, key, **kwargs):
        super().__init__(**kwargs)
        scale_key, mlp_key = jr.split(key)
        if scale:
            self.scale = jr.uniform(scale_key, (hidden_size,), minval=0.9, maxval=1.1)
        else:
            self.scale = 1
        self.mlp = eqx.nn.MLP(
            in_size=hidden_size + 1,
            out_size=hidden_size,
            width_size=width_size,
            depth=depth,
            activation=lipswish,
            final_activation=jnn.tanh,
            key=mlp_key,
        )

    def __call__(self, t, y, args):
        t = jnp.asarray(t)
        return self.scale * self.mlp(jnp.concatenate([t[None], y]))


class PosteriorField(eqx.Module):
    """Posterior drift that conditions on encoder context passed via args=(ts, ctx)."""

    scale: jnp.ndarray
    mlp: eqx.nn.MLP

    def __init__(
        self, latent_size, context_size, width_size, depth, scale, *, key, **kwargs
    ):
        super().__init__(**kwargs)
        scale_key, mlp_key = jr.split(key)
        if scale:
            self.scale = jr.uniform(scale_key, (latent_size,), minval=0.9, maxval=1.1)
        else:
            self.scale = 1
        self.mlp = eqx.nn.MLP(
            in_size=latent_size + context_size + 1,  # [t, y, ctx]
            out_size=latent_size,
            width_size=width_size,
            depth=depth,
            activation=lipswish,
            final_activation=jnn.tanh,
            key=mlp_key,
        )

    def __call__(self, t, y, args):
        t = jnp.asarray(t)
        ts, ctx = args
        i = jnp.minimum(jnp.searchsorted(ts, t, side="right"), ts.shape[0] - 1)
        return self.scale * self.mlp(jnp.concatenate([t[None], y, ctx[i]]))


class Encoder(eqx.Module):
    gru: eqx.nn.GRUCell
    lin: eqx.nn.Linear
    hidden_size: int = eqx.field(static=True)

    def __init__(self, data_size: int, hidden_size: int, ctx_size: int, *, key) -> None:
        gru_key, lin_key = jr.split(key)
        self.hidden_size = hidden_size
        self.gru = eqx.nn.GRUCell(
            input_size=data_size, hidden_size=hidden_size, key=gru_key
        )
        self.lin = eqx.nn.Linear(hidden_size, ctx_size, key=lin_key)

    def __call__(self, xs: jnp.ndarray) -> jnp.ndarray:
        """Encode sequence xs: (T, data_size) -> (T, ctx_size)."""

        def step(h, x):
            h = self.gru(x, h)
            return h, self.lin(h)

        _, ctx = jax.lax.scan(step, jnp.zeros(self.hidden_size), xs)
        return ctx

In [ ]:
class LatentSDE(eqx.Module):
    encoder: Encoder
    f: PosteriorField  # Posterior drift (uses context via args)
    h: VectorField  # Prior drift
    g: VectorField  # Shared diagonal diffusion
    qz0_posterior: eqx.nn.Linear
    pz0_mean: jnp.ndarray
    pz0_logvar: jnp.ndarray
    log_sigma: jnp.ndarray
    decoder: eqx.nn.MLP
    latent_size: int = eqx.field(static=True)
    data_size: int = eqx.field(static=True)

    def __init__(
        self,
        data_size: int,
        latent_size: int,
        context_size: int,
        hidden_size: int,
        *,
        key,
        **kwargs,
    ) -> None:
        super().__init__(**kwargs)
        (enc_key, qz0_key, f_key, h_key, g_key, dec_key, mean_key, logvar_key) = (
            jr.split(key, 8)
        )

        self.data_size = data_size
        self.latent_size = latent_size

        self.encoder = Encoder(data_size, hidden_size, context_size, key=enc_key)
        self.qz0_posterior = eqx.nn.Linear(context_size, 2 * latent_size, key=qz0_key)

        self.f = PosteriorField(
            latent_size, context_size, hidden_size, 4, scale=True, key=f_key
        )
        self.h = VectorField(latent_size, hidden_size, 4, scale=True, key=h_key)
        self.g = VectorField(latent_size, hidden_size, 4, scale=True, key=g_key)

        self.decoder = eqx.nn.MLP(latent_size, data_size, hidden_size, 1, key=dec_key)

        self.pz0_mean = jr.normal(mean_key, (latent_size,)) * 0.1
        self.pz0_logvar = jr.uniform(
            logvar_key, (latent_size,), minval=-0.5, maxval=0.5
        )

        self.log_sigma = jnp.full((data_size,), -1.0)

    def __call__(
        self, xs: jnp.ndarray, ts: jnp.ndarray, *, key
    ) -> tuple[jnp.ndarray, jnp.ndarray]:
        bm_key, z0_key = jr.split(key)

        # Encode context (reverse time GRU, flip back)
        ctx = jnp.flip(self.encoder(jnp.flip(xs, axis=0)), axis=0)  # (T, ctx_size)

        # SDE integration setup
        t0, t1, dt0 = ts[0], ts[-1], 1.0
        bm = diffrax.VirtualBrownianTree(
            t0, t1, tol=dt0 / 2, shape=(self.latent_size + 1,), key=bm_key
        )

        drift_term = diffrax.ODETerm(self.drift)
        diffusion_term = diffrax.ControlTerm(self.diffusion, bm)
        sde = diffrax.MultiTerm(drift_term, diffusion_term)
        solver = diffrax.Euler()
        saveat = diffrax.SaveAt(ts=ts)

        # Sample initial condition from posterior
        qz0_mean, qz0_logstd = jnp.split(self.qz0_posterior(ctx[0]), 2, axis=-1)
        qz0_logstd = jnp.clip(qz0_logstd, -5.0, 2.0)
        z0 = qz0_mean + jnp.exp(qz0_logstd) * jr.normal(z0_key, shape=qz0_mean.shape)

        # Augmented initial state: [z0, 0] (KL accumulator starts at 0)
        z0_aug = jnp.concatenate([z0, jnp.zeros(1)])

        # Context passed via args so PosteriorField can access it (no mutation)
        sol = diffrax.diffeqsolve(
            sde, solver, t0, t1, dt0, z0_aug, saveat=saveat, args=(ts, ctx)
        )
        zs = sol.ys[..., :-1]  # (T, latent_size)
        logqp_path = sol.ys[-1, -1]  # Accumulated path KL

        # Decode latent path to observation space
        xs_hat = jax.vmap(self.decoder)(zs)  # (T, data_size)

        # --- Log-likelihood of observations ---
        sigma = jnp.exp(jnp.clip(self.log_sigma, -4.0, 2.0))
        log_pxs = (
            jax.scipy.stats.norm.logpdf(xs, xs_hat, sigma).sum(axis=-1).mean()
        )  # sum features, mean time

        # --- KL divergence of initial condition ---
        pz0_logstd = jnp.clip(self.pz0_logvar * 0.5, -5.0, 2.0)
        kl_z0 = (
            pz0_logstd
            - qz0_logstd
            + (jnp.exp(2 * qz0_logstd) + (qz0_mean - self.pz0_mean) ** 2)
            / (2 * jnp.exp(2 * pz0_logstd))
            - 0.5
        )
        logqp0 = kl_z0.sum(axis=-1)  # sum over latent dims

        return log_pxs, logqp0 + logqp_path

    def sample(self, ts: jnp.ndarray, *, key) -> jnp.ndarray:
        """Sample from the prior SDE (no observations needed).
        Returns decoded trajectory, shape (T, data_size).
        """
        bm_key, z0_key = jr.split(key)

        t0, t1, dt0 = ts[0], ts[-1], 1.0
        bm = diffrax.VirtualBrownianTree(
            t0, t1, tol=dt0 / 2, shape=(self.latent_size,), key=bm_key
        )

        def prior_drift(t, y, args):
            return self.h(t, y, args)

        def prior_diffusion(t, y, args):
            return jnp.diag(self.g(t, y, args))

        terms = diffrax.MultiTerm(
            diffrax.ODETerm(prior_drift),
            diffrax.ControlTerm(prior_diffusion, bm),
        )
        solver = diffrax.Euler()
        saveat = diffrax.SaveAt(ts=ts)

        # Sample z0 from prior
        pz0_std = jnp.exp(jnp.clip(self.pz0_logvar * 0.5, -5.0, 2.0))
        z0 = self.pz0_mean + pz0_std * jr.normal(z0_key, shape=self.pz0_mean.shape)

        sol = diffrax.diffeqsolve(terms, solver, t0, t1, dt0, z0, saveat=saveat)
        return jax.vmap(self.decoder)(sol.ys)  # (T, data_size)

    def drift(self, t: jnp.ndarray, y: jnp.ndarray, args) -> jnp.ndarray:
        y_state = y[..., :-1]  # Strip KL accumulator
        z_f = self.f(t, y_state, args)
        z_h = self.h(t, y_state, args)
        z_g = self.g(t, y_state, args)
        u = (z_f - z_h) / jnp.clip(jnp.abs(z_g), min=1e-7)
        return jnp.concatenate([z_f, jnp.array([0.5 * jnp.sum(u**2)])])

    def diffusion(self, t: jnp.ndarray, y: jnp.ndarray, args) -> jnp.ndarray:
        y_state = y[..., :-1]
        g_val = self.g(t, y_state, args)
        # Diagonal matrix for augmented state: [g(z), 0]
        return jnp.diag(jnp.concatenate([g_val, jnp.zeros(1)]))

In [ ]:
@eqx.filter_jit
def loss_fn(
    sde: LatentSDE, xs: jnp.ndarray, ts: jnp.ndarray, *, key, kl_weight: float = 1.0
) -> jnp.ndarray:
    """ELBO loss: -log p(x|z) + kl_weight * KL.

    Args:
        sde: LatentSDE model (single sample, no batch dim).
        xs: Observations, shape (T, data_size).
        ts: Timestamps, shape (T,).
        key: PRNG key.
        kl_weight: Annealing weight for KL term (0 → 1 over training).
    Returns:
        Scalar loss (negative ELBO).
    """
    log_pxs, kl = sde(xs, ts, key=key)
    return -log_pxs + kl_weight * kl


@eqx.filter_jit
def batch_loss_fn(
    sde: LatentSDE,
    xs_batch: jnp.ndarray,
    ts: jnp.ndarray,
    *,
    key,
    kl_weight: float = 1.0,
) -> jnp.ndarray:
    """Mean loss over a batch. xs_batch shape: (batch, T, data_size)."""
    batch_size = xs_batch.shape[0]
    keys = jr.split(key, batch_size)
    losses = jax.vmap(lambda x, k: loss_fn(sde, x, ts, key=k, kl_weight=kl_weight))(
        xs_batch, keys
    )
    return jnp.mean(losses)


def increase_update_initial(updates, sde):
    """Multiply gradient updates for initial condition params (pz0_mean, pz0_logvar, qz0_posterior) by 10."""
    initial_leaves = lambda u: [
        u.pz0_mean,
        u.pz0_logvar,
        u.qz0_posterior.weight,
        u.qz0_posterior.bias,
    ]
    return eqx.tree_at(initial_leaves, updates, replace_fn=lambda x: x * 10)


@eqx.filter_jit
def train_step(
    sde: LatentSDE, opt_state, optimizer, xs_batch, ts, *, key, kl_weight: float = 1.0
):
    loss, grads = eqx.filter_value_and_grad(batch_loss_fn)(
        sde, xs_batch, ts, key=key, kl_weight=kl_weight
    )
    grads = increase_update_initial(grads, sde)
    updates, opt_state = optimizer.update(grads, opt_state, sde)
    sde = eqx.apply_updates(sde, updates)
    return sde, opt_state, loss

In [ ]:
def generate_ou_data(
    key,
    dim: int = 12,
    t0: float = 0.0,
    t1: float = 10.0,
    dt0: float = 0.01,
    theta: jnp.ndarray | None = None,
    mu: jnp.ndarray | None = None,
    sigma: jnp.ndarray | None = None,
):
    """Generate a single d-dimensional Ornstein-Uhlenbeck sample with per-dimension parameters.

    If theta/mu/sigma are None, random per-dimension values are drawn:
      - theta ~ LogUniform[0.001, 0.1]  (mean-reversion speed)
      - mu    ~ Uniform[-5, 5]          (long-run mean)
      - sigma ~ LogUniform[0.05, 1.0]   (volatility)
    """
    bm_key, y0_key, param_key = jr.split(key, 3)
    tk, mk, sk = jr.split(param_key, 3)

    if theta is None:
        theta = jnp.exp(
            jr.uniform(tk, (dim,), minval=jnp.log(0.001), maxval=jnp.log(0.1))
        )
    if mu is None:
        mu = jr.uniform(mk, (dim,), minval=-5.0, maxval=5.0)
    if sigma is None:
        sigma = jnp.exp(
            jr.uniform(sk, (dim,), minval=jnp.log(0.05), maxval=jnp.log(1.0))
        )

    def ou_drift(t, y, args):
        return theta * (mu - y)

    def ou_diffusion(t, y, args):
        return jnp.diag(sigma)

    bm = diffrax.UnsafeBrownianPath(shape=(dim,), key=bm_key)
    terms = diffrax.MultiTerm(
        diffrax.ODETerm(ou_drift),
        diffrax.ControlTerm(ou_diffusion, bm),
    )
    solver = diffrax.Euler()
    saveat = diffrax.SaveAt(ts=jnp.arange(t0, t1, dt0))

    y0 = jr.normal(y0_key, (dim,))
    sol = diffrax.diffeqsolve(
        terms,
        solver,
        t0,
        t1,
        dt0,
        y0,
        saveat=saveat,
        max_steps=None,
        adjoint=diffrax.ForwardMode(),
    )
    return sol.ts, sol.ys

In [ ]:
# --- Hyperparameters ---
import os

DATA_SIZE = 12
LATENT_SIZE = 8  # compress 12→8: positions & velocities share structure
CONTEXT_SIZE = 64
HIDDEN_SIZE = 128
LR_INIT = 1e-3  # conservative start; SDE training is sensitive
LR_GAMMA = 0.999  # gentler decay (~0.6× after 500 epochs)
NUM_EPOCHS = 2000  # start shorter, extend once loss stabilises
KL_ANNEAL_ITERS = 500  # ramp KL over ~500 epochs to let decoder learn first
BATCH_SIZE = 32  # moderate batch for stable gradients without OOM
DT = 0.05
SAMPLE_LENGTH = 5000  # 250 s windows — manageable for memory & gradients
LOG_EVERY = 10
SAMPLE_EVERY = 250
CHECKPOINT_EVERY = 250  # save model every N epochs

FEATS = [
    "pos_eta_x",
    "pos_eta_y",
    "pos_eta_mz",
    "pos_nu_x",
    "pos_nu_y",
    "pos_nu_mz",
    "rpm_bow_fore",
    "rpm_bow_aft",
    "rpm_stern_fore",
    "rpm_stern_aft",
    "rpm_fixed_ps",
    "rpm_fixed_sb",
]

if "DATABRICKS_RUNTIME_VERSION" in os.environ:
    DATA_PATH = Path("/Volumes/main_udev/ai_labs/aag/data/pilot_3dof/")
    CHECKPOINT_DIR = Path(
        "/Volumes/main_udev/ai_labs/aag/artifacts/checkpoints/latentsde_diffrax"
    )
else:
    DATA_PATH = Path(
        r"C:\Users\AAg\OneDrive - Allseas Engineering BV\Documents\Thesis\data"
    )
    CHECKPOINT_DIR = Path("runs/latentSDE/checkpoints")

CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)

# --- Data loading ---
files = find_parquet_files(
    DATA_PATH,
    lambda m: m["end_time"] == 10800 and m["timestep"] == 0.05,
)
dataset = ParquetDataset(
    files, columns=FEATS, sample_length=SAMPLE_LENGTH, standardise=True
)
dataloader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=0)
print(
    f"Loaded {len(files)} files, {len(dataset)} samples, {len(dataloader)} batches/epoch"
)

# --- Model + optimizer ---
key = jax.random.key(7777)
model_key, train_key = jr.split(key)

sde = LatentSDE(DATA_SIZE, LATENT_SIZE, CONTEXT_SIZE, HIDDEN_SIZE, key=model_key)

schedule = optax.exponential_decay(LR_INIT, transition_steps=1, decay_rate=LR_GAMMA)
optimizer = optax.adam(schedule)
opt_state = optimizer.init(eqx.filter(sde, eqx.is_array))

In [ ]:
# --- Training loop ---
import tempfile


with mlflow.start_run(run_name=f"latentsde_diffrax_bs{BATCH_SIZE}_lr{LR_INIT}"):
    mlflow.log_params(
        {
            "model_type": "LatentSDE_diffrax",
            "batch_size": BATCH_SIZE,
            "latent_size": LATENT_SIZE,
            "context_size": CONTEXT_SIZE,
            "hidden_size": HIDDEN_SIZE,
            "lr_init": LR_INIT,
            "lr_gamma": LR_GAMMA,
            "num_epochs": NUM_EPOCHS,
            "kl_anneal_iters": KL_ANNEAL_ITERS,
            "dt": DT,
            "sample_length": SAMPLE_LENGTH,
            "n_files": len(files),
        }
    )

    global_step = 0
    best_loss = float("inf")

    for epoch in range(NUM_EPOCHS):
        epoch_key = jr.fold_in(train_key, epoch)
        kl_weight = min(1.0, epoch / KL_ANNEAL_ITERS)
        epoch_losses = []

        for i, (ts_batch, xs_batch, meta) in enumerate(dataloader):
            # Validate aligned timesteps across the batch
            t = ts_batch[0]
            if not torch.allclose(
                ts_batch, t.unsqueeze(0).expand_as(ts_batch), atol=0.005
            ):
                print(f"Skipping batch {i} — misaligned timesteps")
                continue

            # Skip incomplete batches (last batch may be smaller)
            if xs_batch.shape[0] != BATCH_SIZE:
                continue

            # Convert to JAX arrays: ts (T,), xs (batch, T, data_size)
            ts = jnp.asarray(t.numpy())
            xs = jnp.asarray(xs_batch.numpy())

            step_key = jr.fold_in(epoch_key, i)
            sde, opt_state, loss = train_step(
                sde,
                opt_state,
                optimizer,
                xs,
                ts,
                key=step_key,
                kl_weight=kl_weight,
            )

            loss_val = float(loss)
            epoch_losses.append(loss_val)

            if global_step % LOG_EVERY == 0:
                lr_now = float(schedule(global_step))
                mlflow.log_metrics(
                    {
                        "batch_loss": loss_val,
                        "learning_rate": lr_now,
                        "kl_weight": kl_weight,
                    },
                    step=global_step,
                )

            global_step += 1

        # Epoch summary
        if epoch_losses:
            mean_loss = sum(epoch_losses) / len(epoch_losses)
            print(
                f"Epoch {epoch:05d} | mean_loss: {mean_loss:.4f} | kl_w: {kl_weight:.3f}"
            )
            mlflow.log_metric("epoch_loss", mean_loss, step=epoch)

            # Save best model
            if mean_loss < best_loss:
                best_loss = mean_loss
                with tempfile.TemporaryDirectory() as tmpdir:
                    tmp_path = Path(tmpdir) / f"best_{epoch:05d}.eqx"
                    eqx.tree_serialise_leaves(tmp_path, sde)
                    mlflow.log_metric("best_loss", best_loss, step=epoch)
                    mlflow.log_artifact(str(tmp_path), artifact_path="best")

        # Periodic checkpoint
        if epoch % CHECKPOINT_EVERY == 0 and epoch > 0:
            with tempfile.TemporaryDirectory() as tmpdir:
                tmp_path = Path(tmpdir) / f"checkpoint_{epoch:05d}.eqx"
                eqx.tree_serialise_leaves(tmp_path, sde)

                mlflow.log_artifact(str(tmp_path), artifact_path="checkpoint")

        # Sample from prior and log figures
        if epoch % SAMPLE_EVERY == 0 and epoch > 0:
            sample_key = jr.fold_in(epoch_key, 999)
            sample_keys = jr.split(sample_key, 4)
            sample_ts = jnp.arange(0, SAMPLE_LENGTH * DT, DT)
            samples = jax.vmap(lambda k: sde.sample(sample_ts, key=k))(sample_keys)
            # samples: (4, T, data_size)

            fig, axes = plt.subplots(3, 4, figsize=(16, 9), sharex=True)
            for j, ax in enumerate(axes.flat):
                for s in range(samples.shape[0]):
                    ax.plot(sample_ts, samples[s, :, j], alpha=0.6, linewidth=0.5)
                ax.set_title(FEATS[j])
            fig.suptitle(f"Prior samples — Epoch {epoch}")
            plt.tight_layout()
            mlflow.log_figure(fig, f"samples/epoch_{epoch:05d}.png")
            plt.close(fig)

    # Save final model + log as artifact
    final_path = CHECKPOINT_DIR / "final.eqx"
    eqx.tree_serialise_leaves(final_path, sde)
    mlflow.log_artifact(str(final_path), artifact_path="checkpoints")
    mlflow.log_artifact(str(CHECKPOINT_DIR / "best.eqx"), artifact_path="checkpoints")
    print(f"Training complete. Best loss: {best_loss:.4f}")